# Build and evaluate models 



## read energy weather Germany data from saved csv file

In [ ]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
import pandas as pd

from fetch_prepare_data import *
from train_model_predict import *

# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})


In [ ]:
import pandas as pd

df_energy_weather = pd.read_csv('../data/processed/energy_weather_2019_2025.csv', parse_dates=['DateUTC'])

display(df_energy_weather.head())
print(df_energy_weather.columns)

## Train, test data split

* split into train and test sets based on date till end of 2024 for training, and 2025 for testing

In [ ]:
train_data = df_energy_weather[df_energy_weather['DateUTC'] < '2025-01-01']
test_data = df_energy_weather[df_energy_weather['DateUTC'] >= '2025-01-01']

features_train = train_data.drop(['DateUTC', 'EnergyDemand'], axis=1)
target_train = train_data['EnergyDemand']
features_test = test_data.drop(['DateUTC', 'EnergyDemand'], axis=1)
target_test = test_data['EnergyDemand']

print(f"Training data shape: {features_train.shape}")
print(f"Testing data shape: {features_test.shape}")

# Build column transformer
* Standard Scaler for numerical columns: 
        'holiday_ratio', 'apparent_temperature', 'rain', 'snowfall', 'wind_speed_10m', 'shortwave_radiation',  'apparent_temperature_rolling_mean_24h', 'apparent_temperature_lag_24h', 'shortwave_radiation_0m_rolling_mean_24h',       'shortwave_radiation_0m_lag_24h', 'heating_degree', 'cooling_degree'
* OneHotEncoder for categorical columns: 'year', 'hour', 'weekday', 'month', 'is_weekend', 'is_holiday', 'is_pandemic_time'

In [ ]:
with open('../data/processed/energy_weather_data_for_modeling.pkl', 'rb') as f:
    data_for_modeling = pickle.load(f)
features_train, target_train, features_test, target_test = train_test_split_by_date(data_for_modeling, 
                                                                                    date_column='time',
                                                                                    target_column='EnergyDemand', 
                                                                                    split_date='2025-01-01')

In [ ]:
# column transformer pipeline for preprocessing
from sklearn.compose import ColumnTransformer   
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# identify numeric and categorical columns
numeric_features = features_train.select_dtypes(include=['float64']).columns
categorical_features = features_train.select_dtypes(include=['int64']).columns
# numeric transformer
numeric_transformer = Pipeline(steps=[ 
    ('scaler', StandardScaler()) 
])  
# categorical transformer
categorical_transformer = Pipeline(steps=[  
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])  

# combine transformers into a column transformer
preprocessor = ColumnTransformer(   
    transformers=[ 
        ('num', numeric_transformer, numeric_features), 
        ('cat', categorical_transformer, categorical_features) 
    ])



# Build basic models for ML
* Linear Regression
* Random Forest Regression
* Support Vector Regression - not suitable for such large number of data points
* XGBoost  

In [ ]:
# Linear Regression pipeline
from sklearn.linear_model import LinearRegression  
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
from fetch_prepare_data import *
import numpy as np

lr_pipeline = Pipeline(steps=[ 
    ('preprocessor', preprocessor), 
    ('model', LinearRegression()) 
])
lr_pipeline.fit(features_train, target_train)
lr_predictions = lr_pipeline.predict(features_test)

print_scores('Linear Regression', target_test, lr_predictions)

In [ ]:
# Random Forest model
from sklearn.ensemble import RandomForestRegressor  

rf_model = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
rf_model.fit(features_train, target_train)
rf_predictions = rf_model.predict(features_test) 

print_scores('Random Forest', target_test, rf_predictions)


In [ ]:
# Support Vector Regression pipeline for not suitable for large datasets, so we will just use a subset of the data for testing
features_train_subset = features_train.sample(n=10000, random_state=42)
target_train_subset = target_train.loc[features_train_subset.index]

from sklearn.svm import SVR

svr_pipeline = Pipeline(steps=[ 
    ('preprocessor', preprocessor), 
    ('model', SVR(kernel='rbf')) 
])
svr_pipeline.fit(features_train_subset, target_train_subset)
svr_predictions = svr_pipeline.predict(features_test)   

print_scores('Support Vector Regression', target_test, svr_predictions)

In [ ]:
!pip install xgboost
#uv add baysian-optimization 

In [ ]:
# XGBoost model
from xgboost import XGBRegressor    

xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(features_train, target_train)
xgb_predictions = xgb_model.predict(features_test)

print_scores('XGBoost', target_test, xgb_predictions)

In [ ]:
!pip install lightgbm

In [ ]:
# LightGBM model - tree based, no need for column transformer, can handle categorical features directly
from lightgbm import LGBMRegressor  
lgbm_model = LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, force_col_wise=True)
lgbm_model.fit(features_train, target_train)
lgbm_predictions = lgbm_model.predict(features_test)

print_scores('LightGBM', target_test, lgbm_predictions)

In [ ]:
# suppress warnings for SARIMAX 
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

# SARIMAX model - time series model, can handle seasonality and exogenous variables (weather features)
features_train_sarimax, target_train_sarimax, features_test_sarimax, target_test_sarimax = train_test_split_by_date_for_sarimax(data_for_modeling, 
                                                                                    date_column='time',
                                                                                    target_column='EnergyDemand', 
                                                                                    split_date='2025-01-01')
features_train_sarimax.set_index(features_train_sarimax['time'], inplace=True)
features_train_sarimax.drop('time', axis=1, inplace=True)
features_test_sarimax.set_index(features_test_sarimax['time'], inplace=True)
features_test_sarimax.drop('time', axis=1, inplace=True)
target_train_sarimax.set_index(target_train_sarimax['time'], inplace=True)
target_train_sarimax.drop('time', axis=1, inplace=True)
target_test_sarimax.set_index(target_test_sarimax['time'], inplace=True)
target_test_sarimax.drop('time', axis=1, inplace=True)

# reduce data to daily frequency by resampling and taking the mean, since SARIMAX can be very slow on large datasets
target_train_sarimax = target_train_sarimax.resample('D').mean()
features_train_sarimax = features_train_sarimax.resample('D').mean()
target_test_sarimax = target_test_sarimax.resample('D').mean()
features_test_sarimax = features_test_sarimax.resample('D').mean()

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_model = SARIMAX(
    target_train_sarimax, 
    exog=features_train_sarimax, 
    order=(1, 0, 1), 
    seasonal_order=(1, 0, 1, 24))
sarimax_results = sarimax_model.fit(disp=False)

# use integer indices for predict to avoid date-matching issues with out-of-sample data
start_idx = len(target_train_sarimax)
end_idx = start_idx + len(target_test_sarimax) - 1
sarimax_predictions = sarimax_results.predict(start=start_idx, end=end_idx, exog=features_test_sarimax)
sarimax_predictions.index = target_test_sarimax.index  # restore date index for evaluation

print_scores('SARIMAX', target_test_sarimax, sarimax_predictions)


# Tune the hyperparameters on selected models

* selected models: Random Forest 
* Use RandomizedSearchCV - samples fixed number of random combinations to reducing effort/time
* SARIMAX on daily data aggregation reached the best score
- SARIMAX MAE: 468.4341511754603
- SARIMAX MSE: 351086.6416544649
- SARIMAX R2: 0.9917514259109218



In [ ]:

# tune Random Forest hyperparameters using RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

# use TimeSeriesSplit for cross-validation to respect the temporal order of the data
tscv = TimeSeriesSplit(n_splits=5)

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],          # lower depth = less overfitting
    'min_samples_split': [2, 5, 10], # min samples to split a node
    'min_samples_leaf': [1, 2, 4],   # min samples in a leaf
    'max_features': [1.0, 'sqrt', 'log2'], # 1.0 replaces deprecated 'auto' (use all features)
}
rf_random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42), 
    param_distributions=rf_param_grid, 
    n_iter=20, 
    cv=tscv, 
    scoring='neg_mean_absolute_error', #'neg_root_mean_squared_error get worse MAE and MSE, but better R2, so we will stick with MAE for tuning
    verbose=2, 
    random_state=42, 
    n_jobs=-1)
rf_random_search.fit(features_train, target_train)
best_rf_model = rf_random_search.best_estimator_  

rf_tuned_predictions = best_rf_model.predict(features_test)

print_scores('Tuned Random Forest', target_test, rf_tuned_predictions)


In [ ]:
# tune XGBoost hyperparameters using RandomizedSearchCV
from xgboost import XGBRegressor    
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
xgb_random_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42), 
    param_distributions=xgb_param_grid, 
    n_iter=20, 
    cv=tscv, 
    scoring='neg_mean_absolute_error', 
    verbose=2, 
    random_state=42, 
    n_jobs=-1)      

xgb_random_search.fit(features_train, target_train)
best_xgb_model = xgb_random_search.best_estimator_
xgb_tuned_predictions = best_xgb_model.predict(features_test)

print_scores('Tuned XGBoost', target_test, xgb_tuned_predictions)


In [ ]:
# tune LightGBM hyperparameters using RandomizedSearchCV
# use TimeSeriesSplit for cross-validation to respect the temporal order of the data
tscv = TimeSeriesSplit(n_splits=5)

lgbm_param_grid = { 
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 9]
}   
lgbm_random_search = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42, force_col_wise=True),   
    param_distributions=lgbm_param_grid,
    n_iter=10,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    verbose=4,
    random_state=42,
    n_jobs=-1
    )
lgbm_random_search.fit(features_train, target_train)
best_lgbm_model = lgbm_random_search.best_estimator_    
lgbm_tuned_predictions = best_lgbm_model.predict(features_test)

print_scores('Tuned LightGBM', target_test, lgbm_tuned_predictions)

In [ ]:
# save models to Pickle files for later use
import pickle   
with open('../models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)
with open('../models/best_lgbm_model.pkl', 'wb') as f:
    pickle.dump(best_lgbm_model, f)

In [ ]:
# plot the predictions vs actual values for the best random forest model
plt.figure(figsize=(16, 5)) 
plt.plot(test_data['DateUTC'], target_test, label='Actual', color='blue')
plt.plot(test_data['DateUTC'], rf_tuned_predictions, label='Predicted', color='orange')
plt.xlabel('Date')
plt.ylabel('Energy Demand')
plt.title('Random Forest Predictions vs Actual')  
plt.legend()
plt.grid()  
plt.show()


In [ ]:
# plot the predictions vs actual values for the best LightGBM model
plt.figure(figsize=(16, 5)) 
plt.plot(test_data['DateUTC'], target_test, label='Actual', color='blue')
plt.plot(test_data['DateUTC'], lgbm_tuned_predictions, label='Predicted', color='orange')
plt.xlabel('Date')
plt.ylabel('Energy Demand')
plt.title('LightGBM Predictions vs Actual')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import pickle 
import datetime
import matplotlib.pyplot as plt

# predict the next 24 hours of energy demand using the best LightGBM model
best_lgbm_model = pickle.load(open('../models/best_lgbm_model.pkl', 'rb'))
future_features = features_test.tail(24)  # use the last 24 hours of features for prediction
future_predictions = best_lgbm_model.predict(future_features)
print("Predicted energy demand for the next 24 hours:")
print(future_predictions)

file_scrapted_smard = pd.read_csv('../data/raw/netzlast_2025-10-01_to_2026-05-10_20.csv')
file_scrapted_smard['timestamp'] = pd.to_datetime(file_scrapted_smard['timestamp'], utc=True)  # ensure timestamp is in datetime format

# read the actual values for 2025-10-01 from the scraped data to compare with the predictions
actual_values_2025_10_01 = file_scrapted_smard[file_scrapted_smard['timestamp'].dt.date == pd.to_datetime('2025-10-01').date()]['load_MWh'].values
print("Actual energy demand for 2025-10-01:")
print(actual_values_2025_10_01)

# compare the future predictions to the actual values for 2025-10-01
actual_future = actual_values_2025_10_01
plt.figure(figsize=(10, 5))
plt.plot(range(len(actual_future)), actual_future, label='Actual', color='blue')       
plt.plot(range(len(future_predictions)), future_predictions, label='Predicted', color='orange')
plt.xlabel('Date')
plt.ylabel('Energy Demand')
plt.title('Future Predictions vs Actual with LightGBM')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# predict the next 24 hours of energy demand using the best Random Forest model
best_rf_model = pickle.load(open('../models/best_rf_model.pkl', 'rb'))
future_features = features_test.tail(24)  # use the last 24 hours of features for prediction
future_predictions = best_rf_model.predict(future_features)
print("Predicted energy demand for the next 24 hours:")
print(future_predictions)

file_scrapted_smard = pd.read_csv('../data/raw/netzlast_2025-10-01_to_2026-05-10_20.csv')
file_scrapted_smard['timestamp'] = pd.to_datetime(file_scrapted_smard['timestamp'], utc=True)  # ensure timestamp is in datetime format

# read the actual values for 2025-10-01 from the scraped data to compare with the predictions
actual_values_2025_10_01 = file_scrapted_smard[file_scrapted_smard['timestamp'].dt.date == pd.to_datetime('2025-10-01').date()]['load_MWh'].values
print("Actual energy demand for 2025-10-01:")
print(actual_values_2025_10_01)

# compare the future predictions to the actual values for 2025-10-01
actual_future = actual_values_2025_10_01
plt.figure(figsize=(10, 5))
plt.plot(range(len(actual_future)), actual_future, label='Actual', color='blue')       
plt.plot(range(len(future_predictions)), future_predictions, label='Predicted', color='orange')
plt.xlabel('Date')
plt.ylabel('Energy Demand')
plt.title('Future Predictions vs Actual with Random Forest')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# predict the next 7 days of energy demand using the best LightGBM model
future_features = features_test.tail(24)  # use the last 24 hours of features for prediction
future_predictions = best_lgbm_model.predict(future_features)
print("Predicted energy demand for the next 24 hours:")
print(future_predictions)

file_scrapted_smard = pd.read_csv('../data/raw/netzlast_2025-10-01_to_2026-05-10_20.csv')
file_scrapted_smard['timestamp'] = pd.to_datetime(file_scrapted_smard['timestamp'], utc=True)  # ensure timestamp is in datetime format

# read the actual values for 2025-10-01 to 2025-10-07 from the scraped data to compare with the predictions
filter_timestamp = (file_scrapted_smard['timestamp'].dt.date >= pd.to_datetime('2025-10-01').date()) & (file_scrapted_smard['timestamp'].dt.date < pd.to_datetime('2025-10-08').date())
actual_values_2025_10_01_to_2025_10_07 = file_scrapted_smard[filter_timestamp]['load_MWh'].values
print("Actual energy demand for 2025-10-01 to 2025-10-07:")
print(actual_values_2025_10_01_to_2025_10_07)

# compare the future predictions to the actual values for 2025-10-01 to 2025-10-07
actual_future = actual_values_2025_10_01_to_2025_10_07
plt.figure(figsize=(10, 5))
plt.plot(range(len(actual_future)), actual_future, label='Actual', color='blue')       
plt.plot(range(len(future_predictions)), future_predictions, label='Predicted', color='orange')
plt.xlabel('Date')
plt.ylabel('Energy Demand')
plt.title('Future Predictions vs Actual with LightGBM')
plt.legend()
plt.grid()
plt.show()


In [ ]:
future_features[future_features['is_holiday'] == 1]